# Computational Set 13: Newton–Raphson Optimization
[Jay Foley, University of North Carolina Charlotte](https://foleylab.github.io/)

#### Learning Outcomes
By the end of this workbook, students should be able to
- Derive the Newton–Raphson update step from a Taylor expansion in one and several dimensions
- Implement Newton–Raphson optimization for both quadratic and non-quadratic functions
- Generalize the 1D Newton–Raphson step to multiple dimensions using the gradient and Hessian
- Approximate gradients and Hessians using central finite differences
- Apply Newton–Raphson optimization to a real potential energy surface computed with PySCF
- Continue practicing the Design Recipe to compose functions

#### Summary
We will use Python and NumPy (and, later, PySCF) to build up a Newton–Raphson optimizer from scratch, one function at a time. Newton–Raphson is an iterative method for finding a stationary point (a point where the first derivative vanishes) of a function. It is widely used in computational chemistry to optimize molecular geometries: the "function" being optimized is the potential energy of the molecule as a function of its nuclear coordinates, and a stationary point corresponds to an equilibrium geometry.

Just as in [Computational Set 1](spin_one_half.ipynb), we will use the **Design Recipe** to build each new function we need. By the end of the notebook you will have written your own general-purpose 1D and multi-dimensional Newton–Raphson optimizers, and used them to find the equilibrium bond length of carbon monoxide (CO) from an *ab initio* potential energy surface.

# Import statements
We will use `numpy` for numerical computing and `matplotlib` for plotting. Later in the notebook we will also use `pyscf`, a package for quantum chemistry calculations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Functions and the Design Recipe
Recall from [Computational Set 1](spin_one_half.ipynb) that we build functions using the **Design Recipe**:

1. **Header:** Define the function's name, input parameters, and their data types. Specify the data type of the return value.
2. **Purpose:** Write a concise one-line description explaining the function's purpose.
3. **Examples:** Provide examples of how to call the function with different inputs and the expected outputs. These examples will be used to test the function.
4. **Body:** Write the actual code that implements the function's logic, based on the header, purpose, and examples.
5. **Test:** Test your function on all your example cases; try to identify additional tricky cases.
6. **Debug/Iterate:** If your tests fail, re-read your code, check your logic, check your syntax, and fix any mistakes that you catch. If you cannot catch mistakes, think about additional test cases that can help reveal flaws in your logic. Continue until your function passes tests.

In this notebook, most functions will already have their **Header**, **Purpose**, and (at least some) **Examples** written for you. Your job is mostly to complete Steps 4–6: write the **Body**, **Test** it against the examples (and against your own physical/mathematical reasoning), and **Debug/Iterate** until it works. Incomplete lines are marked with `...` — replace each `...` with the appropriate code.

## ⚙️ Part 1: Newton–Raphson in One Dimension

Let's consider the function:

$$
f(x) = (x - 2)^2 - 2 = x^2 - 4x + 2
$$

This function has a stationary point at $x_s = 2$. In the language of geometry optimization, this would be an equilibrium geometry.

### 🧩 Design Recipe: `f`, `fprime`, `fdoubleprime`

We need three small functions: $f(x)$, $f'(x) = 2x-4$, and $f''(x) = 2$.

1. **Header** Each function takes one float `x` and returns one float.
2. **Purpose** Given below in each docstring.
3. **Examples** Given below in each docstring; add a third example of your own to each docstring.
4. **Body** Complete the line marked `...` in each function.
5. **Test** Run the test cell below and confirm all three functions pass.
6. **Debug/Iterate** If a test fails, recheck your algebra before you recheck your code.

In [ ]:
def f(x):
    """
    Evaluate f(x) = x^2 - 4x + 2.

    Arguments
    ---------
    x : float
        The point at which to evaluate f.

    Returns
    -------
    f_x : float
        The value of f(x).

    Examples
    --------
    f(0) == 2
    f(2) == -2
    # TODO: add a third example
    """
    f_x = x**2 - 4*x + 2
    return f_x


def fprime(x):
    """
    Evaluate the first derivative f'(x) = 2x - 4.

    Arguments
    ---------
    x : float
        The point at which to evaluate f'.

    Returns
    -------
    fp_x : float
        The value of f'(x).

    Examples
    --------
    fprime(0) == -4
    fprime(2) == 0
    # TODO: add a third example
    """
    fp_x = 2*x - 4
    return fp_x


def fdoubleprime(x):
    """
    Evaluate the second derivative f''(x) = 2.

    Arguments
    ---------
    x : float
        The point at which to evaluate f''.

    Returns
    -------
    fpp_x : float
        The value of f''(x), which is constant for this function.

    Examples
    --------
    fdoubleprime(0) == 2
    fdoubleprime(100) == 2
    """
    fpp_x = 2
    return fpp_x

In [ ]:
# ✅ Tests -- these should all pass once f, fprime, and fdoubleprime are complete
assert np.isclose(f(0), 2)
assert np.isclose(f(2), -2)
assert np.isclose(fprime(0), -4)
assert np.isclose(fprime(2), 0)
assert np.isclose(fdoubleprime(0), 2)
assert np.isclose(fdoubleprime(100), 2)
print("✅ f, fprime, and fdoubleprime all passed their tests!")

### 🧩 Design Recipe: `newton_raphson_step`

From the Taylor expansion of $f$ about a point $x_n$, the Newton–Raphson step toward a stationary point is:

$$
x_{n+1} = x_n - \frac{f'(x_n)}{f''(x_n)}
$$

Let's build a single-step function that we can reuse throughout the rest of the notebook.

1. **Header** `newton_raphson_step(fp_val, fpp_val, x_val)` takes three floats (the first derivative, the second derivative, and the current point) and returns one float (the updated point).
2. **Purpose** Given in the docstring below.
3. **Examples** Two examples are given below — work through them by hand before running the test cell.
4. **Body** Complete the line marked `...`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If your test fails, double check the sign in the update formula.

In [ ]:
def newton_raphson_step(fp_val, fpp_val, x_val):
    """
    Compute one Newton-Raphson update toward a stationary point.

    Arguments
    ---------
    fp_val : float
        The first derivative evaluated at the current point, f'(x_val).
    fpp_val : float
        The second derivative evaluated at the current point, f''(x_val).
    x_val : float
        The current point, x_val.

    Returns
    -------
    x_new : float
        The updated point, x_val - fp_val / fpp_val.

    Examples
    --------
    newton_raphson_step(-2, 2, 1.0) == 2.0
    newton_raphson_step(-10, 2, 0.0) == 5.0
    """
    x_new = x_val - fp_val / fpp_val
    return x_new

In [ ]:
# ✅ Tests
assert np.isclose(newton_raphson_step(-2, 2, 1.0), 2.0)
assert np.isclose(newton_raphson_step(-10, 2, 0.0), 5.0)
print("✅ newton_raphson_step passed both tests!")

### Applying `newton_raphson_step` to our quadratic function

We'll start at an initial guess $x_0 = 1$ and take a single Newton–Raphson step toward the stationary point of $f(x) = x^2-4x+2$.

**Your task:** use `f`, `fprime`, `fdoubleprime`, and `newton_raphson_step` to evaluate $f(x_0)$, $f'(x_0)$, $f''(x_0)$, and the updated point $x_1$.

In [ ]:
x0 = 1.0

# TODO: use the functions above to evaluate f, f', and f'' at x0
f_x0 = f(x0)
fp_x0 = fprime(x0)
fpp_x0 = fdoubleprime(x0)

# TODO: use newton_raphson_step to compute x1 from fp_x0, fpp_x0, and x0
x1 = newton_raphson_step(fp_x0, fpp_x0, x0)

print(f"f({x0}) = {f_x0}")
print(f"f'({x0}) = {fp_x0}")
print(f"f''({x0}) = {fpp_x0}")
print(f"Newton-Raphson step gives x1 = {x1}")

# ✅ Test -- because f is exactly quadratic, x1 should land exactly on the stationary point x_s = 2
assert np.isclose(x1, 2.0), "x1 should equal the true stationary point, x_s = 2"
print("✅ x1 matches the true stationary point!")

**Question:** Because our function is perfectly quadratic, the Newton–Raphson step finds the stationary point in a **single iteration**. Why does this happen for quadratic functions specifically? *Hint: think about what a second-order Taylor expansion of a quadratic function looks like.*

In later sections, we'll see what happens when $f(x)$ isn't purely quadratic.

### 🎨 Visualizing the Newton–Raphson step

Run the cell below (no changes needed) to see the function, the initial point $x_0$, and the point $x_1$ found by the Newton–Raphson step.

In [ ]:
x_plot = np.linspace(-1, 4, 200)
y_plot = f(x_plot)

# Tangent line scaled by 1/f''(x0), passing through (x0, f(x0))
y_tangent = f_x0 + fp_x0 / fpp_x0 * (x_plot - x0)

plt.figure(figsize=(6, 4))
plt.plot(x_plot, y_plot, label=r"$f(x) = x^2 - 4x + 2$")
plt.plot(x_plot, y_tangent, '--', label=r"Tangent scaled by $1/f''(x_0)$")
plt.scatter([x0], [f_x0], color='red', zorder=3, label=r"$x_0$")
plt.scatter([x1], [f(x1)], color='green', zorder=3, label=r"$x_1$ (stationary point)")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Newton-Raphson Step for a Quadratic Function")
plt.legend()
plt.grid(True)
plt.show()

## 🚀 Part 2: Newton–Raphson for a Non-Quadratic Function

Now let's consider a function that is **not** quadratic:

$$
f(x) = x^3 + x^2 - 5
$$

The stationary points are at $x_s = 0$ and $x_s = -\tfrac{2}{3}$, but imagine we don't know these ahead of time. Since $f$ is no longer quadratic, its curvature ($f''$) changes with $x$, so we expect to need **more than one** Newton–Raphson step to converge.

Below, $f$, $f'$, and $f''$ for this new function are already implemented for you, since you just practiced writing this kind of function in Part 1.

In [ ]:
def f(x):
    """Return f(x) = x^3 + x^2 - 5."""
    return x**3 + x**2 - 5


def fprime(x):
    """Return f'(x) = 3x^2 + 2x."""
    return 3*x**2 + 2*x


def fdoubleprime(x):
    """Return f''(x) = 6x + 2."""
    return 6*x + 2

x0 = 1.0
fp_x0 = fprime(x0)
fpp_x0 = fdoubleprime(x0)
x1 = newton_raphson_step(fp_x0, fpp_x0, x0)

print(f"x0 = {x0:.4f}")
print(f"x1 = {x1:.4f}")

**Question:** How close is $x_1$ to one of the true stationary points ($x_s = 0$ or $x_s = -2/3$)? You should find that one iteration isn't enough this time, since the function isn't quadratic.

### 🧩 Design Recipe: `newton_raphson_1d`

Rather than manually repeating `newton_raphson_step` over and over, let's write a function that iterates automatically until convergence.

1. **Header** `newton_raphson_1d(fprime, fdoubleprime, x0, tol=1e-6, max_iter=20)` takes two functions (`fprime` and `fdoubleprime`), an initial guess `x0`, a convergence tolerance `tol`, and a maximum number of iterations `max_iter`. It returns a tuple `(x_opt, n_iter)`: the converged point and the number of iterations used.
2. **Purpose** Given in the docstring below.
3. **Examples** Two examples are given below.
4. **Body** Complete the loop, using `newton_raphson_step` at each iteration. Stop iterating once the size of the step, $|\Delta x|$, is smaller than `tol`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** A common bug here is forgetting to update `x` at the end of each loop iteration — if your function never converges, that's the first thing to check.

In [ ]:
def newton_raphson_1d(fprime, fdoubleprime, x0, tol=1e-6, max_iter=20):
    """
    Iteratively apply the Newton-Raphson update until convergence.

    Arguments
    ---------
    fprime : callable
        Function that returns f'(x) given x.
    fdoubleprime : callable
        Function that returns f''(x) given x.
    x0 : float
        Initial guess for the stationary point.
    tol : float, optional
        Convergence tolerance on the step size |x_new - x| (default 1e-6).
    max_iter : int, optional
        Maximum number of iterations to attempt (default 20).

    Returns
    -------
    x_opt : float
        The converged stationary point (or the last point reached if
        max_iter was exceeded without convergence).
    n_iter : int
        The number of iterations actually used.

    Examples
    --------
    # A quadratic function converges in 2 iterations (evaluated at x=0, then x=2)
    newton_raphson_1d(lambda x: 2*x - 4, lambda x: 2, x0=0.0)[0]  # -> 2.0

    # x^3 + x^2 - 5 started near x=1 converges toward the stationary point x_s = 0
    newton_raphson_1d(lambda x: 3*x**2 + 2*x, lambda x: 6*x + 2, x0=1.0)[0]  # -> approx 0.0
    """
    x = x0
    for i in range(max_iter):
        fp_val = fprime(x)
        fpp_val = fdoubleprime(x)
        x_new = newton_raphson_step(fp_val, fpp_val, x)

        step_size = abs(x_new - x)

        if step_size < tol:
            return x_new, i + 1

        x = x_new

    return x, max_iter

In [ ]:
# ✅ Tests
x_opt_quad, n_iter_quad = newton_raphson_1d(lambda x: 2*x - 4, lambda x: 2, x0=0.0)
assert np.isclose(x_opt_quad, 2.0)
assert n_iter_quad <= 3, "The quadratic case should converge in just a couple of iterations"

x_opt_cubic, n_iter_cubic = newton_raphson_1d(lambda x: 3*x**2 + 2*x, lambda x: 6*x + 2, x0=1.0)
assert np.isclose(x_opt_cubic, 0.0, atol=1e-5)

print(f"Quadratic case converged to x = {x_opt_quad:.6f} in {n_iter_quad} iterations")
print(f"Cubic case (x0=1.0) converged to x = {x_opt_cubic:.6f} in {n_iter_cubic} iterations")
print("✅ newton_raphson_1d passed both tests!")

### 🧩 Practice Problem

**Your task:** Use `newton_raphson_1d` to find the *other* stationary point of $f(x) = x^3+x^2-5$ by starting from a different initial guess (try `x0 = -1.0`).

<details>
<summary><strong>💡 Solution</strong></summary>

```python
x_opt_2, n_iter_2 = newton_raphson_1d(lambda x: 3*x**2 + 2*x, lambda x: 6*x + 2, x0=-1.0)
print(x_opt_2)  # approximately -0.6667, i.e. -2/3
```

Starting from `x0 = -1.0` (which is closer to $x_s = -2/3$ than to $x_s = 0$), Newton–Raphson converges to the *other* stationary point, $x_s \approx -2/3$. This illustrates an important property of Newton's method: it converges to whichever stationary point is "downhill" in curvature from the initial guess, which is not always the globally nearest one, and is not guaranteed to be a minimum (as opposed to a maximum or saddle point).
</details>

In [ ]:
# TODO: call newton_raphson_1d starting from x0 = -1.0 and store the result
x_opt_2, n_iter_2 = newton_raphson_1d(lambda x: 3*x**2 + 2*x, lambda x: 6*x + 2, x0=-1.0)
print(f"Starting from x0 = -1.0, converged to x = {x_opt_2:.6f} in {n_iter_2} iterations")

assert np.isclose(x_opt_2, -2/3, atol=1e-4)
print("✅ Found the other stationary point!")

## 🧭 Part 3: Newton–Raphson in Two Dimensions

In one dimension, we wrote the Newton–Raphson update for finding a stationary point as

$$
x_{n+1} = x_n - \frac{f'(x_n)}{f''(x_n)}.
$$

In **two dimensions (and higher)**, this generalizes to:

$$
\mathbf{x}_{n+1} = \mathbf{x}_n - \mathbf{H}^{-1}(\mathbf{x}_n) \, \nabla f(\mathbf{x}_n)
$$

where $\nabla f(\mathbf{x})$ is the **gradient vector** and $\mathbf{H}(\mathbf{x})$ is the **Hessian matrix** (matrix of second derivatives). In practice, we avoid explicitly forming $\mathbf{H}^{-1}$ and instead solve the linear system $\mathbf{H} \, \Delta\mathbf{x} = -\nabla f$ for $\Delta \mathbf{x}$, which is both faster and numerically more stable.

We'll begin with a perfectly quadratic function:

$$
f(x, y) = (x - 1)^2 + 2(y - 2)^2
$$

which has its stationary point (and minimum) at $(x_s, y_s) = (1, 2)$.

### 🧩 Design Recipe: `f_quad`, `grad_quad`, `hessian_quad`

1. **Header** `f_quad(x, y)` returns a float. `grad_quad(x, y)` returns a length-2 NumPy array $[\partial f/\partial x, \partial f/\partial y]$. `hessian_quad(x, y)` returns a $2\times2$ NumPy array of second partial derivatives.
2. **Purpose** Given in each docstring.
3. **Examples** Given in each docstring.
4. **Body** Complete each line marked `...`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** Double check your partial derivatives by hand if a test fails.

In [ ]:
def f_quad(x, y):
    """
    Evaluate f(x, y) = (x-1)^2 + 2(y-2)^2.

    Arguments
    ---------
    x, y : float
        The coordinates at which to evaluate f.

    Returns
    -------
    fxy : float
        The value of f(x, y).

    Examples
    --------
    f_quad(1, 2) == 0
    f_quad(0, 0) == 1 + 8
    """
    fxy = (x - 1)**2 + 2*(y - 2)**2
    return fxy


def grad_quad(x, y):
    """
    Evaluate the gradient of f_quad: [df/dx, df/dy].

    Arguments
    ---------
    x, y : float
        The coordinates at which to evaluate the gradient.

    Returns
    -------
    grad : ndarray of shape (2,)
        The gradient vector [2(x-1), 4(y-2)].

    Examples
    --------
    np.allclose(grad_quad(1, 2), [0, 0])
    np.allclose(grad_quad(0, 0), [-2, -8])
    """
    dfdx = 2*(x - 1)
    dfdy = 4*(y - 2)
    return np.array([dfdx, dfdy])


def hessian_quad(x, y):
    """
    Evaluate the Hessian of f_quad.

    Arguments
    ---------
    x, y : float
        The coordinates at which to evaluate the Hessian
        (unused here, since the Hessian of this function is constant,
        but included for a consistent function signature).

    Returns
    -------
    H : ndarray of shape (2, 2)
        The (constant) Hessian matrix [[2, 0], [0, 4]].

    Examples
    --------
    np.allclose(hessian_quad(0, 0), [[2, 0], [0, 4]])
    np.allclose(hessian_quad(5, -3), [[2, 0], [0, 4]])
    """
    H = np.array([[2.0, 0.0], [0.0, 4.0]])
    return H

In [ ]:
# ✅ Tests
assert np.isclose(f_quad(1, 2), 0)
assert np.isclose(f_quad(0, 0), 9)
assert np.allclose(grad_quad(1, 2), [0, 0])
assert np.allclose(grad_quad(0, 0), [-2, -8])
assert np.allclose(hessian_quad(0, 0), [[2, 0], [0, 4]])
assert np.allclose(hessian_quad(5, -3), [[2, 0], [0, 4]])
print("✅ f_quad, grad_quad, and hessian_quad all passed their tests!")

### Applying a single 2D Newton–Raphson step

Starting from $\mathbf{x}_0 = (0, 0)$, compute one Newton update using `np.linalg.solve` to solve $\mathbf{H}\,\Delta\mathbf{x} = -\nabla f$ for $\Delta\mathbf{x}$, then set $\mathbf{x}_1 = \mathbf{x}_0 + \Delta\mathbf{x}$.

In [ ]:
xy0 = np.array([0.0, 0.0])

# TODO: evaluate the gradient and Hessian of f_quad at xy0
g = grad_quad(*xy0)
H = hessian_quad(*xy0)

# TODO: solve H @ delta = -g for delta, then compute xy1 = xy0 + delta
delta = np.linalg.solve(H, -g)
xy1 = xy0 + delta

print("x0 =", xy0)
print("Gradient =", g)
print("x1 =", xy1)

# ✅ Test -- because f_quad is exactly quadratic, xy1 should land exactly on (1, 2)
assert np.allclose(xy1, [1.0, 2.0])
print("✅ Landed exactly on the stationary point (1, 2)!")

**Question:** Why does it only take one step to reach the stationary point for this function, just like in the 1D quadratic case?

### 🧩 Design Recipe: `newton_raphson_2d`

Now let's write a general iterative optimizer that works for **any** 2D function, given its gradient and Hessian.

1. **Header** `newton_raphson_2d(grad_f, hessian_f, xy0, tol=1e-6, max_iter=20)` takes the gradient function, the Hessian function, an initial point `xy0` (a length-2 array), a tolerance, and a maximum number of iterations. It returns `(xy_opt, n_iter)`.
2. **Purpose** Given in the docstring below.
3. **Examples** Given in the docstring below.
4. **Body** Complete the loop, reusing the `np.linalg.solve` approach from above. Stop when $\|\Delta\mathbf{x}\|$ (its Euclidean norm, `np.linalg.norm`) is smaller than `tol`.
5. **Test** Run the test cell below.
6. **Debug/Iterate** If your function raises a `LinAlgError`, check that your Hessian function is returning a $2\times2$ array (not a 1D array or scalar).

In [ ]:
def newton_raphson_2d(grad_f, hessian_f, xy0, tol=1e-6, max_iter=20):
    """
    Iteratively apply the multi-dimensional Newton-Raphson update until convergence.

    Arguments
    ---------
    grad_f : callable
        Function that takes (x, y) and returns the gradient as a length-2 array.
    hessian_f : callable
        Function that takes (x, y) and returns the Hessian as a 2x2 array.
    xy0 : array-like of shape (2,)
        Initial guess [x0, y0].
    tol : float, optional
        Convergence tolerance on the norm of the step, ||delta|| (default 1e-6).
    max_iter : int, optional
        Maximum number of iterations to attempt (default 20).

    Returns
    -------
    xy_opt : ndarray of shape (2,)
        The converged point (or the last point reached if max_iter was
        exceeded without convergence).
    n_iter : int
        The number of iterations actually used.

    Examples
    --------
    # The quadratic case converges in a small number of iterations to (1, 2)
    newton_raphson_2d(grad_quad, hessian_quad, [0.0, 0.0])[0]  # -> array([1., 2.])
    """
    xy = np.array(xy0, dtype=float)
    for i in range(max_iter):
        g = grad_f(*xy)
        H = hessian_f(*xy)
        delta = np.linalg.solve(H, -g)  # TODO: solve H @ delta = -g for delta
        xy_new = xy + delta  # TODO: update xy using delta

        step_norm = np.linalg.norm(delta)  # TODO: compute the Euclidean norm of delta

        if step_norm < tol:
            return xy_new, i + 1

        xy = xy_new

    return xy, max_iter

In [ ]:
# ✅ Tests
xy_opt, n_iter = newton_raphson_2d(grad_quad, hessian_quad, [0.0, 0.0])
assert np.allclose(xy_opt, [1.0, 2.0])
assert n_iter <= 3

print(f"Converged to {xy_opt} in {n_iter} iterations")
print("✅ newton_raphson_2d passed the quadratic test!")

### A non-quadratic 2D function

Now let's consider a *non-quadratic* function, where the curvature changes with position:

$$
f(x, y) = \sin(x) + \cos(y)
$$

This function has stationary points where both partial derivatives vanish:

$$
\frac{\partial f}{\partial x} = \cos(x) = 0, \qquad \frac{\partial f}{\partial y} = -\sin(y) = 0,
$$

i.e. where $x = \pi/2 + n\pi$ and $y = n\pi$ for integer $n$.

**Your task:** Complete `f2`, `grad_f2`, and `hessian_f2` below, following the same pattern as `f_quad`, `grad_quad`, and `hessian_quad`.

In [ ]:
def f2(x, y):
    """
    Evaluate f2(x, y) = sin(x) + cos(y).

    Examples
    --------
    np.isclose(f2(0, 0), 1.0)
    """
    fxy = np.sin(x) + np.cos(y)
    return fxy


def grad_f2(x, y):
    """
    Evaluate the gradient of f2: [cos(x), -sin(y)].

    Examples
    --------
    np.allclose(grad_f2(0, 0), [1.0, 0.0])
    """
    dfdx = np.cos(x)
    dfdy = -np.sin(y)
    return np.array([dfdx, dfdy])


def hessian_f2(x, y):
    """
    Evaluate the Hessian of f2: [[-sin(x), 0], [0, -cos(y)]].

    Examples
    --------
    np.allclose(hessian_f2(0, 0), [[0.0, 0.0], [0.0, -1.0]])
    """
    d2fx2 = -np.sin(x)
    d2fxy = 0.0
    d2fyx = 0.0
    d2fy2 = -np.cos(y)
    return np.array([[d2fx2, d2fxy],
                     [d2fyx, d2fy2]])

In [ ]:
# ✅ Tests
assert np.isclose(f2(0, 0), 1.0)
assert np.allclose(grad_f2(0, 0), [1.0, 0.0])
assert np.allclose(hessian_f2(0, 0), [[0.0, 0.0], [0.0, -1.0]])
print("✅ f2, grad_f2, and hessian_f2 all passed their tests!")

# Apply our general 2D optimizer, starting from (2.0, 1.0)
xy_opt2, n_iter2 = newton_raphson_2d(grad_f2, hessian_f2, [2.0, 1.0])
print(f"Converged to {xy_opt2} in {n_iter2} iterations, f2 = {f2(*xy_opt2):.4f}")

**Questions:**
1. Which stationary point does the iteration converge to, in terms of $\pi/2 + n\pi$ and $n\pi$?
2. What happens if you start from a different initial guess, e.g. $(x_0, y_0) = (0, 0)$? *(Try it in a new cell!)*
3. Does the algorithm always converge to a minimum, or can it find a maximum or saddle point?

### 💡 Discussion

In 2D, the Newton–Raphson method uses the Hessian matrix to locally approximate the curvature in all directions. If the Hessian is positive definite, you're near a **minimum**; if it's negative definite, you're near a **maximum**; and if it has both positive and negative eigenvalues, you've found a **saddle point**. You can check which case you're in using `np.linalg.eigh` on the Hessian at the converged point.

## ⚙️ Part 4: Finite-Difference Gradients and Hessians

So far, we've relied on **analytical derivatives** — but in many real problems (including quantum chemistry calculations), we can't easily compute $\nabla f$ or $\mathbf{H}$ analytically. A common alternative is to use **finite differences**:

$$
\frac{\partial f}{\partial x_i} \approx \frac{f(x_i + h) - f(x_i - h)}{2h}
$$

and

$$
\frac{\partial^2 f}{\partial x_i \partial x_j} \approx
\frac{f(x_i + h, x_j + h) - f(x_i + h, x_j - h)
- f(x_i - h, x_j + h) + f(x_i - h, x_j - h)}{4h^2}
$$

where $h$ is a small step along the direction $x_i$ or $x_j$. This approximation becomes more accurate as $h \to 0$, so numerically we typically set $h$ to some small value like $h = 10^{-3}$.

We'll test this idea using $f_2(x, y) = \sin(x) + \cos(y)$ and compare our **numerical** gradients and Hessians to the **analytical** ones from Part 3.

### 🧩 Design Recipe: `numerical_grad` and `numerical_hessian`

1. **Header** `numerical_grad(f, x, h=1e-3)` takes a function `f` of a NumPy array, a point `x` (an array), and a step size `h`; it returns the gradient as an array the same shape as `x`. `numerical_hessian(f, x, h=1e-4)` similarly returns the Hessian as a square array.
2. **Purpose** Given in each docstring.
3. **Examples** Given in each docstring.
4. **Body** Complete the central-difference formulas marked `...`.
5. **Test** Run the test cell below, comparing against `grad_f2` and `hessian_f2`.
6. **Debug/Iterate** Remember that `f` here takes two *separate* arguments (x and y), so call it as `f(*x)` when `x` is a length-2 array.

In [ ]:
def numerical_grad(f, x, h=1e-3):
    """
    Compute the numerical gradient of a function using central finite differences.

    Arguments
    ---------
    f : callable
        A function that takes N separate scalar arguments, e.g. f(x, y).
    x : array-like of shape (N,)
        The point at which to evaluate the gradient.
    h : float, optional
        The finite-difference step size (default 1e-3).

    Returns
    -------
    grad : ndarray of shape (N,)
        The numerical gradient vector.

    Examples
    --------
    np.allclose(numerical_grad(f2, [0.0, 0.0]), [1.0, 0.0], atol=1e-4)
    """
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x)

    for i in range(len(x)):
        step = np.zeros_like(x)
        step[i] = h

        f_plus = f(*(x + step))   # TODO: evaluate f at x + step
        f_minus = f(*(x - step))  # TODO: evaluate f at x - step
        grad[i] = (f_plus - f_minus) / (2 * h)  # TODO: central-difference formula

    return grad


def numerical_hessian(f, x, h=1e-4):
    """
    Compute the numerical Hessian of a function using central finite differences.

    Arguments
    ---------
    f : callable
        A function that takes N separate scalar arguments, e.g. f(x, y).
    x : array-like of shape (N,)
        The point at which to evaluate the Hessian.
    h : float, optional
        The finite-difference step size (default 1e-4).

    Returns
    -------
    H : ndarray of shape (N, N)
        The numerical Hessian matrix.

    Examples
    --------
    np.allclose(numerical_hessian(f2, [0.0, 0.0]), [[0.0, 0.0], [0.0, -1.0]], atol=1e-3)
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            ei = np.zeros_like(x)
            ej = np.zeros_like(x)
            ei[i] = h
            ej[j] = h

            f_pp = f(*(x + ei + ej))  # TODO: f(x + ei + ej)
            f_pm = f(*(x + ei - ej))  # TODO: f(x + ei - ej)
            f_mp = f(*(x - ei + ej))  # TODO: f(x - ei + ej)
            f_mm = f(*(x - ei - ej))  # TODO: f(x - ei - ej)

            H[i, j] = (f_pp - f_pm - f_mp + f_mm) / (4 * h**2)  # TODO: 4-point central-difference formula

    return H

In [ ]:
def f2_wrapped(*args):
    return f2(*args)

# ✅ Tests -- compare numerical derivatives to the analytical ones from Part 3
grad_numeric = numerical_grad(f2_wrapped, [0.3, 0.7])
grad_analytic = grad_f2(0.3, 0.7)
assert np.allclose(grad_numeric, grad_analytic, atol=1e-4), (grad_numeric, grad_analytic)

hess_numeric = numerical_hessian(f2_wrapped, [0.3, 0.7])
hess_analytic = hessian_f2(0.3, 0.7)
assert np.allclose(hess_numeric, hess_analytic, atol=1e-3), (hess_numeric, hess_analytic)

print("Numerical gradient: ", grad_numeric, " Analytical:", grad_analytic)
print("Numerical Hessian:\n", hess_numeric, "\nAnalytical:\n", hess_analytic)
print("✅ numerical_grad and numerical_hessian agree with the analytical derivatives!")

## ⚛️ Part 5: Geometry Optimization of CO Using PySCF

So far, we've used simple mathematical functions to test our Newton–Raphson and finite-difference code. Let's now connect this to a real chemistry application: computing the **potential energy surface** of carbon monoxide (CO) using *ab initio* methods, and using our own `newton_raphson_1d`-style logic (adapted for numerical derivatives) to find its equilibrium bond length.

### Installing PySCF
If PySCF is not already installed in your environment, run `!pip install pyscf` in a new cell, or install it into your `chem5200` conda environment with `pip install pyscf` (PySCF is not currently packaged on conda-forge, so `pip` is the recommended install method even inside a conda environment).

### ✅ Worked example: `compute_pyscf_energy`

This function is a bit more involved because it uses the PySCF API, so it is given to you fully worked out below — but it still follows the Design Recipe! Look for the **Header** (name, arguments, return type), **Purpose** (docstring first line), and **Body** (the implementation), and notice how the **Examples** in the docstring double as documentation for anyone reading your code later.

In [ ]:
from pyscf import gto, scf

# Conversion factor: Angstrom -> Bohr (atomic units)
ang_to_au = 1.88973

# Template for the CO molecule; the C atom sits at the origin and the O
# atom is placed a variable distance away along x.
mol_template = """
C 0 0 0
O {} 0 0
"""

def compute_pyscf_energy(R_CO_angstrom):
    """
    Compute the RHF total energy (in Hartree) for CO at a given C-O distance.

    Arguments
    ---------
    R_CO_angstrom : float
        The C-O bond length in Angstroms.

    Returns
    -------
    energy : float
        The Hartree-Fock total energy (in Hartree atomic units).

    Examples
    --------
    compute_pyscf_energy(1.13)  # approximately -112.75 Hartree, near the CO equilibrium bond length
    """
    mol = gto.M(
        atom=mol_template.format(R_CO_angstrom),
        basis='cc-pvdz',
        verbose=0,
        spin=0
    )
    mf = scf.RHF(mol)
    mf.kernel()
    return mf.e_tot

### Potential energy scan

Run the cell below to compute and plot the RHF energy of CO as a function of bond length.

In [ ]:
bond_lengths = np.arange(0.8, 1.6, 0.01)
hf_energies = [compute_pyscf_energy(R) for R in bond_lengths]

print("First few RHF energies (Hartree):")
print(hf_energies[:5])

plt.plot(bond_lengths, hf_energies, label="RHF Energy")
plt.xlabel("Bond length (Angstroms)")
plt.ylabel("Energy (Hartree)")
plt.legend()
plt.show()

**Question:** What trend do you see in the energy values as the bond length changes? Roughly where is the equilibrium bond length?

### 🧩 Design Recipe: `numerical_gradient_energy` and `numerical_second_derivative`

We'll reuse the central-difference idea from Part 4, but now applied to the PySCF energy function, which only depends on a single variable (the bond length $R$). Note that PySCF takes $R$ in Angstroms but returns the energy in Hartree (atomic units of energy), so to get a derivative in consistent atomic units (Hartree / Bohr) we need to convert the finite-difference step size $h$ from Angstroms to Bohr using `ang_to_au = 1.88973`.

1. **Header** `numerical_gradient_energy(f, R, h=0.01)` and `numerical_second_derivative(f, R, h=0.01)` both take an energy function `f`, a bond length `R` (Angstroms), and a step size `h` (Angstroms); they return the first and second derivatives (Hartree/Bohr and Hartree/Bohr², respectively).
2. **Purpose** Given in each docstring.
3. **Examples** Try `R = 1.2` once your functions are complete, and compare against the trend you saw in the plot above.
4. **Body** Complete the central-difference formulas, remembering to convert `h` to Bohr for the denominator.
5. **Test** Run the cells below.
6. **Debug/Iterate** If your derivative has the wrong sign, double-check which side of the finite-difference formula is `E_plus` and which is `E_minus`.

In [ ]:
def numerical_gradient_energy(f, R, h=0.01):
    """
    Compute the numerical derivative of energy with respect to bond length,
    using a central finite-difference formula.

    Arguments
    ---------
    f : callable
        A function that returns the molecular energy at a given bond length (Angstroms).
    R : float
        Current bond length in Angstroms.
    h : float, optional
        Step size in Angstroms (default 0.01 A).

    Returns
    -------
    dE_dR : float
        Numerical derivative of energy with respect to R, in Hartree per Bohr.
    """
    E_plus = f(R + h)   # TODO: f(R + h)
    E_minus = f(R - h)  # TODO: f(R - h)

    h_au = h * ang_to_au  # convert step size to Bohr for the denominator
    dE_dR = (E_plus - E_minus) / (2 * h_au)  # TODO: central-difference formula
    return dE_dR


def numerical_second_derivative(f, R, h=0.01):
    """
    Compute the numerical second derivative of energy with respect to bond
    length, using the central finite-difference formula.

    Arguments
    ---------
    f : callable
        Function returning molecular energy at a given bond length (Angstroms).
    R : float
        Current bond length (Angstroms).
    h : float, optional
        Step size for finite difference (default 0.01 A).

    Returns
    -------
    d2E_dR2 : float
        Numerical second derivative, in Hartree per Bohr^2.
    """
    E_plus = f(R + h)   # TODO: f(R + h)
    E_minus = f(R - h)  # TODO: f(R - h)
    E0 = f(R)       # TODO: f(R)

    h_au = h * ang_to_au  # convert step size to Bohr for the denominator
    d2E_dR2 = (E_plus - 2*E0 + E_minus) / (h_au**2)  # TODO: central second-difference formula
    return d2E_dR2

In [ ]:
# ✅ Tests -- the gradient should be negative to the left of the minimum
# and positive to the right of it (based on the plot above)
grad_left = numerical_gradient_energy(compute_pyscf_energy, 1.0)
grad_right = numerical_gradient_energy(compute_pyscf_energy, 1.3)
assert grad_left < 0, "Expected a negative slope to the left of the minimum"
assert grad_right > 0, "Expected a positive slope to the right of the minimum"

hess_mid = numerical_second_derivative(compute_pyscf_energy, 1.15)
assert hess_mid > 0, "Expected positive curvature (concave up) near the minimum"

print(f"dE/dR at R=1.00 A: {grad_left:.6f} Hartree/Bohr")
print(f"dE/dR at R=1.30 A: {grad_right:.6f} Hartree/Bohr")
print(f"d2E/dR2 at R=1.15 A: {hess_mid:.6f} Hartree/Bohr^2")
print("✅ numerical_gradient_energy and numerical_second_derivative behave as expected!")

### 🧩 Design Recipe: `newton_optimize_bond`

Finally, let's assemble everything into a 1D Newton–Raphson geometry optimizer for the CO bond length:

$$
R_{n+1} = R_n - \frac{dE/dR}{d^2E/dR^2}
$$

This function is provided complete below, since it mainly just calls the functions you've already written and built the design recipe for above -- but read through it carefully and make sure you can explain what each line does.

In [ ]:
def newton_optimize_bond(f, R0, tol=1e-5, max_iter=10, h=0.01):
    """
    Perform a 1D Newton-Raphson optimization of a bond length.

    Arguments
    ---------
    f : callable
        Function returning molecular energy at a given bond length (Angstroms).
    R0 : float
        Initial bond length (Angstroms).
    tol : float
        Convergence tolerance on the bond-length step, in Angstroms.
    max_iter : int
        Maximum number of iterations.
    h : float
        Step size (Angstroms) used for the numerical derivatives.

    Returns
    -------
    R_opt : float
        Optimized bond length (Angstroms).
    E_opt : float
        Energy at the optimized geometry (Hartree).
    """
    au_to_ang = 1 / ang_to_au
    R = R0
    print(f"Starting optimization from R = {R0:.3f} A")

    for i in range(max_iter):
        grad = numerical_gradient_energy(f, R, h)
        hess = numerical_second_derivative(f, R, h)

        step_au = -grad / hess       # Newton step, in Bohr
        step_ang = step_au * au_to_ang  # convert step to Angstroms
        R_new = R + step_ang

        print(f"Iter {i:2d}: R = {R:.4f} A, E = {f(R):.6f} Ha, dE/dR = {grad:.3e}, step = {step_ang:.3e} A")

        if abs(step_ang) < tol:
            print(f"\n✅ Converged to R = {R_new:.4f} A, E = {f(R_new):.6f} Ha")
            return R_new, f(R_new)

        R = R_new

    print("\n⚠️ Did not converge within max_iter")
    return R, f(R)

In [ ]:
R_start = 1.2  # initial guess, Angstroms
R_opt, E_opt = newton_optimize_bond(compute_pyscf_energy, R_start)

assert 1.0 < R_opt < 1.3, "The optimized CO bond length should be close to 1.11 A"
print(f"\n✅ Optimized R = {R_opt:.4f} A, E = {E_opt:.6f} Ha")

**Questions to explore:**
1. How many iterations did it take to converge?
2. Try changing the initial guess to `R_start = 1.4`, and then to `R_start = 1.5` (use a fresh cell so you don't lose the result above). You should see something surprising:
   - From `R_start = 1.4`, the optimizer overshoots and converges to a bond length of about **-1.11 A** rather than +1.11 A. Since the total energy only depends on the *distance* between C and O, this is not physically wrong (it's the mirror image geometry, same bond length, same energy!) — but it's a good illustration of how Newton–Raphson can jump across a symmetric point when the local curvature is very small.
   - From `R_start = 1.5`, the optimizer diverges to a nonsensical bond length entirely. Print out the curvature (`numerical_second_derivative`) at each iteration to see why: it briefly goes *negative*, meaning the local quadratic model of the PES is concave down there rather than concave up, so the "Newton step" actually moves *away* from the minimum.
3. Why does the second derivative (the curvature) need to be positive at the minimum? What does it mean, physically and numerically, when Newton–Raphson is applied at a point where the curvature is close to zero or negative?

## 🌱 Toward Quasi-Newton (BFGS)

Quasi-Newton methods like BFGS avoid explicitly constructing the Hessian matrix, and instead build up an approximation to it using only gradient information collected along the optimization path. You can read more about BFGS [here](https://towardsdatascience.com/bfgs-in-a-nutshell-an-introduction-to-quasi-newton-methods-21b0e13ee504/) and [here](https://en.wikipedia.org/wiki/Broyden%E2%80%93Fletcher%E2%80%93Goldfarb%E2%80%93Shanno_algorithm).

### 🧩 Design Recipe: `bfgs_update` (challenge extension)

1. **Header** What should the function be called? What arguments does it need (position vectors, gradient vectors, an approximate Hessian or its inverse from the previous step)? What should it return?
2. **Purpose** Write a one-sentence description of what the function should do.
3. **Examples** This is genuinely hard to write examples for without doing some research first — consider working through a small worked example from the Wikipedia article by hand.
4. **Body** Implement the BFGS update formula.
5. **Test** Compare your BFGS-based optimizer to `newton_raphson_2d` on the quadratic test function `f_quad` — they should converge to the same point, though not necessarily in the same number of iterations.
6. **Debug/Iterate** Some questions to consider along the way:
   - How many position vectors are needed (just the current one, the current and one past position, or more)?
   - How many gradient vectors are needed?
   - Are you directly building an approximation to the Hessian, or to its inverse? Why might one be more convenient than the other?

In [ ]:
# TODO (challenge): implement bfgs_update(...) following the Design Recipe above
